# Lap 4 - Spark DataFrame

Bài thực hành sử dụng **Spark DataFrame** để phân tích dữ liệu đơn hàng, khách hàng, sản phẩm, seller và review.

> Lưu ý quan trọng: Các file CSV trong bài này dùng dấu phân cách `;`, nên khi đọc bằng Spark phải dùng `.option("sep", ";")`.


In [1]:
import os
import sys
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Sửa lại SPARK_HOME nếu Spark của chị nằm ở thư mục khác
JAVA_HOME = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
SPARK_HOME = r"C:\spark\spark-4.1.2-bin-hadoop3"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ["PATH"] = (
    os.path.join(JAVA_HOME, "bin") + ";" +
    os.path.join(SPARK_HOME, "bin") + ";" +
    os.environ["PATH"]
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("JAVA_HOME exists:", os.path.exists(JAVA_HOME))
print("SPARK_HOME exists:", os.path.exists(SPARK_HOME))
print("spark-submit exists:", os.path.exists(os.path.join(SPARK_HOME, "bin", "spark-submit.cmd")))

subprocess.run(["java", "-version"])

spark = SparkSession.builder \
    .appName("Lap4_Spark_DataFrame") \
    .master("local[1]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.sparkContext.version)


Python: c:\Users\Admin\AppData\Local\Programs\python.exe
JAVA_HOME exists: True
SPARK_HOME exists: True
spark-submit exists: True
Spark version: 4.1.2


In [2]:
def resolve_path(filename):
    """
    Ưu tiên đọc file trong cùng thư mục notebook.
    Nếu chạy trên môi trường khác, có thể sửa BASE_DIR bên dưới.
    """
    candidates = [
        os.path.join(os.getcwd(), filename),
        os.path.join("/mnt/data", filename),
        filename
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename

CUSTOMERS_PATH = resolve_path("Customer_List.csv")
ORDER_ITEMS_PATH = resolve_path("Order_Items.csv")
ORDER_REVIEWS_PATH = resolve_path("Order_Reviews.csv")
ORDERS_PATH = resolve_path("Orders.csv")
PRODUCTS_PATH = resolve_path("Products.csv")

print("Customers:", CUSTOMERS_PATH)
print("Order items:", ORDER_ITEMS_PATH)
print("Order reviews:", ORDER_REVIEWS_PATH)
print("Orders:", ORDERS_PATH)
print("Products:", PRODUCTS_PATH)


Customers: d:\study\Big data\Lap 4\Customer_List.csv
Order items: d:\study\Big data\Lap 4\Order_Items.csv
Order reviews: d:\study\Big data\Lap 4\Order_Reviews.csv
Orders: d:\study\Big data\Lap 4\Orders.csv
Products: d:\study\Big data\Lap 4\Products.csv


## 1. Đọc dữ liệu từ các file CSV, tự suy ra kiểu dữ liệu

In [3]:
def read_csv(path):
    return spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("sep", ";") \
        .option("encoding", "UTF-8") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("multiLine", "true") \
        .csv(path)

customers_df = read_csv(CUSTOMERS_PATH)
order_items_df = read_csv(ORDER_ITEMS_PATH)
order_reviews_df = read_csv(ORDER_REVIEWS_PATH)
orders_df = read_csv(ORDERS_PATH)
products_df = read_csv(PRODUCTS_PATH)

print("Customers schema")
customers_df.printSchema()

print("Orders schema")
orders_df.printSchema()

print("Order items schema")
order_items_df.printSchema()

print("Products schema")
products_df.printSchema()

print("Order reviews schema")
order_reviews_df.printSchema()


Customers schema
root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

Orders schema
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

Order items schema
root
 |-- Order_ID: string (nullable = true)
 |-- O

In [4]:
print("Customers")
customers_df.show(5, truncate=False)

print("Orders")
orders_df.show(5, truncate=False)

print("Order Items")
order_items_df.show(5, truncate=False)

print("Products")
products_df.show(5, truncate=False)

print("Order Reviews")
order_reviews_df.show(5, truncate=False)


Customers
+--------------------------------+--------------------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|Customer_Trx_ID                 |Subscriber_ID                   |Subscribe_Date|First_Order_Date|Customer_Postal_Code|Customer_City|Customer_Country|Customer_Country_Code|Age|Gender|
+--------------------------------+--------------------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|1e959e1f5920cba43823fa9f95673b83|9765e039028279fd2e60bb620a451526|2023-07-08    |2023-07-09      |FR-75005            |Paris        |France          |FR                   |29 |Male  |
|9877437582f263da7d7e30a90c57b8bb|a75e134e7eb6f96e2b0c716ac2a82efb|2024-03-23    |2024-04-11      |PL-00-001           |Warsaw       |Poland          |PL                   |38 |Male  |
|fa6fbbb2080646acae977bf2e44af98b|2fdac27295500e820e43910e9a0aa8d

## Chuẩn hóa kiểu dữ liệu ngày và số

In [6]:
orders_clean_df = orders_df \
    .withColumn("Order_Purchase_Timestamp", F.to_timestamp("Order_Purchase_Timestamp", "yyyy-MM-dd HH:mm")) \
    .withColumn("Order_Approved_At", F.to_timestamp("Order_Approved_At", "yyyy-MM-dd HH:mm")) \
    .withColumn("Order_Delivered_Carrier_Date", F.to_timestamp("Order_Delivered_Carrier_Date", "yyyy-MM-dd HH:mm")) \
    .withColumn("Order_Delivered_Customer_Date", F.to_timestamp("Order_Delivered_Customer_Date", "yyyy-MM-dd HH:mm")) \
    .withColumn("Order_Estimated_Delivery_Date", F.to_timestamp("Order_Estimated_Delivery_Date", "yyyy-MM-dd HH:mm"))

order_items_clean_df = order_items_df \
    .withColumn("Shipping_Limit_Date", F.to_timestamp("Shipping_Limit_Date", "yyyy-MM-dd HH:mm")) \
    .withColumn("Price", F.col("Price").cast("double")) \
    .withColumn("Freight_Value", F.col("Freight_Value").cast("double"))

order_reviews_clean_df = order_reviews_df \
    .withColumn("Review_Score", F.col("Review_Score").cast("int")) \
    .withColumn("Review_Creation_Date", F.to_timestamp("Review_Creation_Date", "yyyy-MM-dd HH:mm")) \
    .withColumn("Review_Answer_Timestamp", F.to_timestamp("Review_Answer_Timestamp", "yyyy-MM-dd HH:mm"))

customers_clean_df = customers_df \
    .withColumn("Subscribe_Date", F.to_date("Subscribe_Date", "yyyy-MM-dd")) \
    .withColumn("First_Order_Date", F.to_date("First_Order_Date", "yyyy-MM-dd")) \
    .withColumn("Age", F.col("Age").cast("int"))

products_clean_df = products_df \
    .withColumn("Product_Weight_Gr", F.col("Product_Weight_Gr").cast("double")) \
    .withColumn("Product_Length_Cm", F.col("Product_Length_Cm").cast("double")) \
    .withColumn("Product_Height_Cm", F.col("Product_Height_Cm").cast("double")) \
    .withColumn("Product_Width_Cm", F.col("Product_Width_Cm").cast("double"))


## 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán

In [7]:
total_orders = orders_clean_df.select("Order_ID").distinct().count()
total_customers = customers_clean_df.select("Customer_Trx_ID").distinct().count()
total_sellers = order_items_clean_df.select("Seller_ID").distinct().count()

summary_df = spark.createDataFrame(
    [
        ("Total Orders", total_orders),
        ("Total Customers", total_customers),
        ("Total Sellers", total_sellers)
    ],
    ["Metric", "Value"]
)

summary_df.show(truncate=False)


+---------------+-----+
|Metric         |Value|
+---------------+-----+
|Total Orders   |99441|
|Total Customers|99442|
|Total Sellers  |3095 |
+---------------+-----+



## 3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp giảm dần

In [8]:
orders_by_country_df = orders_clean_df \
    .join(customers_clean_df, on="Customer_Trx_ID", how="left") \
    .groupBy("Customer_Country", "Customer_Country_Code") \
    .agg(F.countDistinct("Order_ID").alias("Total_Orders")) \
    .orderBy(F.desc("Total_Orders"))

orders_by_country_df.show(50, truncate=False)


+----------------+---------------------+------------+
|Customer_Country|Customer_Country_Code|Total_Orders|
+----------------+---------------------+------------+
|Germany         |DE                   |41754       |
|France          |FR                   |12848       |
|Netherlands     |NL                   |11629       |
|Belgium         |BE                   |5464        |
|Austria         |AT                   |5043        |
|Switzerland     |CH                   |3640        |
|United Kingdom  |GB                   |3382        |
|Poland          |PL                   |2139        |
|Czechia         |CZ                   |2034        |
|Italy           |IT                   |2025        |
|Spain           |ES                   |1651        |
|Portugal        |PT                   |1336        |
|Sweden          |SE                   |975         |
|Denmark         |DK                   |905         |
|Serbia          |RS                   |746         |
|Norway          |NO        

## 4. Phân tích số lượng đơn hàng theo năm, tháng đặt hàng

In [9]:
orders_by_year_month_df = orders_clean_df \
    .withColumn("Order_Year", F.year("Order_Purchase_Timestamp")) \
    .withColumn("Order_Month", F.month("Order_Purchase_Timestamp")) \
    .groupBy("Order_Year", "Order_Month") \
    .agg(F.countDistinct("Order_ID").alias("Total_Orders")) \
    .orderBy(F.asc("Order_Year"), F.desc("Order_Month"))

orders_by_year_month_df.show(100, truncate=False)


+----------+-----------+------------+
|Order_Year|Order_Month|Total_Orders|
+----------+-----------+------------+
|2022      |12         |1           |
|2022      |10         |324         |
|2022      |9          |4           |
|2023      |12         |5673        |
|2023      |11         |7544        |
|2023      |10         |4631        |
|2023      |9          |4285        |
|2023      |8          |4331        |
|2023      |7          |4026        |
|2023      |6          |3245        |
|2023      |5          |3700        |
|2023      |4          |2404        |
|2023      |3          |2682        |
|2023      |2          |1780        |
|2023      |1          |800         |
|2024      |10         |4           |
|2024      |9          |16          |
|2024      |8          |6512        |
|2024      |7          |6292        |
|2024      |6          |6167        |
|2024      |5          |6873        |
|2024      |4          |6939        |
|2024      |3          |7211        |
|2024      |

## 5. Thống kê điểm đánh giá trung bình và số lượng đánh giá theo từng mức

In [10]:
# Xử lý NULL và ngoại lệ: chỉ giữ Review_Score từ 1 đến 5
valid_reviews_df = order_reviews_clean_df \
    .filter(F.col("Review_Score").isNotNull()) \
    .filter((F.col("Review_Score") >= 1) & (F.col("Review_Score") <= 5))

avg_review_score_df = valid_reviews_df \
    .agg(F.round(F.avg("Review_Score"), 2).alias("Average_Review_Score"))

review_count_by_score_df = valid_reviews_df \
    .groupBy("Review_Score") \
    .agg(F.count("*").alias("Total_Reviews")) \
    .orderBy("Review_Score")

avg_review_score_df.show(truncate=False)
review_count_by_score_df.show(truncate=False)


+--------------------+
|Average_Review_Score|
+--------------------+
|4.09                |
+--------------------+

+------------+-------------+
|Review_Score|Total_Reviews|
+------------+-------------+
|1           |11424        |
|2           |3151         |
|3           |8179         |
|4           |19141        |
|5           |57328        |
+------------+-------------+



## 6. Doanh thu năm 2024 theo danh mục sản phẩm

In [11]:
revenue_2024_by_category_df = order_items_clean_df \
    .join(orders_clean_df, on="Order_ID", how="inner") \
    .join(products_clean_df, on="Product_ID", how="left") \
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value")) \
    .filter(F.year("Order_Purchase_Timestamp") == 2024) \
    .groupBy("Product_Category_Name") \
    .agg(
        F.round(F.sum("Revenue"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.count("*").alias("Total_Items")
    ) \
    .orderBy(F.desc("Total_Revenue"))

revenue_2024_by_category_df.show(50, truncate=False)


+---------------------------------------+-------------+------------+-----------+
|Product_Category_Name                  |Total_Revenue|Total_Orders|Total_Items|
+---------------------------------------+-------------+------------+-----------+
|Health_Beauty                          |885191.12    |5402        |5951       |
|Watches_Gifts                          |771986.75    |3493        |3703       |
|Bed_Bath_Table                         |650794.7     |4909        |5884       |
|Sports_Leisure                         |621999.34    |4058        |4527       |
|Computers_Accessories                  |594771.04    |4053        |4708       |
|Housewares                             |491576.96    |3421        |4046       |
|Furniture_Decor                        |476466.13    |3199        |4118       |
|Auto                                   |404210.57    |2457        |2619       |
|Baby                                   |299052.56    |1667        |1776       |
|Cool_Stuff                 

## 7. Sản phẩm có số lượng bán ra cao nhất và điểm đánh giá trung bình theo sản phẩm

In [12]:
product_sales_review_df = order_items_clean_df \
    .join(products_clean_df, on="Product_ID", how="left") \
    .join(valid_reviews_df.select("Order_ID", "Review_Score"), on="Order_ID", how="left") \
    .groupBy("Product_ID", "Product_Category_Name") \
    .agg(
        F.count("*").alias("Total_Sold"),
        F.round(F.avg("Review_Score"), 2).alias("Average_Review_Score")
    ) \
    .orderBy(F.desc("Total_Sold"), F.desc("Average_Review_Score"))

product_sales_review_df.show(20, truncate=False)

print("Sản phẩm bán ra cao nhất:")
product_sales_review_df.limit(1).show(truncate=False)


+--------------------------------+---------------------+----------+--------------------+
|Product_ID                      |Product_Category_Name|Total_Sold|Average_Review_Score|
+--------------------------------+---------------------+----------+--------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|Furniture_Decor      |527       |4.02                |
|99a4788cb24856965c36a24e339b6058|Bed_Bath_Table       |491       |3.9                 |
|422879e10f46682990de24d770e7f83d|Garden_Tools         |487       |3.95                |
|389d119b48cf3043d311335e499d9c6b|Garden_Tools         |392       |4.12                |
|368c6c730842d78016ad823897a372db|Garden_Tools         |391       |3.92                |
|53759a2ecddad2bb87a079a1f1519f73|Garden_Tools         |375       |3.87                |
|d1c427060a0f73f6b889a5c7c61f2ac4|Computers_Accessories|343       |4.19                |
|53b36df67ebb7c41585e8d54d6772e08|Watches_Gifts        |323       |4.19                |
|154e7e31ebfa09220379

## 10. Xếp hạng seller theo tổng doanh thu và số lượng đơn hàng bán được

In [13]:
seller_rank_df = order_items_clean_df \
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value")) \
    .groupBy("Seller_ID") \
    .agg(
        F.round(F.sum("Revenue"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.count("*").alias("Total_Items")
    )

rank_window = Window.orderBy(F.desc("Total_Revenue"), F.desc("Total_Orders"))

seller_rank_df = seller_rank_df \
    .withColumn("Seller_Rank", F.dense_rank().over(rank_window)) \
    .orderBy("Seller_Rank")

seller_rank_df.show(50, truncate=False)


+--------------------------------+-------------+------------+-----------+-----------+
|Seller_ID                       |Total_Revenue|Total_Orders|Total_Items|Seller_Rank|
+--------------------------------+-------------+------------+-----------+-----------+
|4869f7a5dfa277a7dca6462dcf3b52b2|249640.7     |1132        |1156       |1          |
|7c67e1448b00f6e969d365cea6b010ab|239536.44    |982         |1364       |2          |
|53243585a1d6dc2643021fd1853d8905|235856.68    |358         |410        |3          |
|4a3ca9315b744ce9f8e9374361493884|235539.96    |1806        |1987       |4          |
|fa1c13f2614d7b5c4749cbc52fecda94|204084.73    |585         |586        |5          |
|da8622b14eb17ae2831f4ac5b9dab84a|185192.32    |1314        |1551       |6          |
|7e93a43ef30c4f03f38b393420bc753a|182754.05    |336         |340        |7          |
|1025f0e2d44d7041d6cf58b6550e0bfa|172860.69    |915         |1428       |8          |
|7a67c85e85bb2ce8582c35f2203ad736|162648.38    |1160  

## Dừng Spark sau khi chạy xong

In [14]:
spark.stop()
